### Import Dependencies

In [1]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient

In [2]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

### RAG Pipeline

In [3]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [4]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

In [5]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt


def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [6]:
output = rag_pipeline("Do you have any earphones?")

In [7]:
output

{'data_object': RAGGenerationResponse(answer='I don’t have any earphones listed among the available products.'),
 'answer': 'I don’t have any earphones listed among the available products.',
 'question': 'Do you have any earphones?',
 'retrieved_context_ids': ['B07VDPWPLQ',
  'B0C61QHRB6',
  'B09TMYXDMG',
  'B09ZTFPVNB',
  'B09PNST3BD'],
 'retrieved_context': ['The Complete Harry Potter Film Music Collection ',
  'If ',
  'Beatopia[LP] ',
  'Girls, Girls, Girls ',
  'Marry Me Soundtrack ']}

In [8]:
print(output["answer"])

I don’t have any earphones listed among the available products.


### RAG Pipeline with Grounding Context

In [9]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="ID of the item used to answer the question")
    description: str = Field(description="Description of the item used to answer the question")

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")

In [10]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- If you are describing multiple products, list them out as a list.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt


def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "references": answer.references,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [11]:
output = rag_pipeline("Do you have any earphones?")

In [12]:
output

{'data_object': RAGGenerationResponse(answer='I don’t see any earphones listed among the available products.', references=[]),
 'answer': 'I don’t see any earphones listed among the available products.',
 'references': [],
 'question': 'Do you have any earphones?',
 'retrieved_context_ids': ['B07VDPWPLQ',
  'B0C61QHRB6',
  'B09TMYXDMG',
  'B09ZTFPVNB',
  'B09PNST3BD'],
 'retrieved_context': ['The Complete Harry Potter Film Music Collection ',
  'If ',
  'Beatopia[LP] ',
  'Girls, Girls, Girls ',
  'Marry Me Soundtrack ']}

In [13]:
print(output["answer"])

I don’t see any earphones listed among the available products.


In [14]:
output = rag_pipeline("Do you have any earphones?", top_k=10)

In [15]:
output

{'data_object': RAGGenerationResponse(answer='I don’t have any earphones listed among the available products right now.', references=[]),
 'answer': 'I don’t have any earphones listed among the available products right now.',
 'references': [],
 'question': 'Do you have any earphones?',
 'retrieved_context_ids': ['B07VDPWPLQ',
  'B0C61QHRB6',
  'B09TMYXDMG',
  'B09ZTFPVNB',
  'B09PNST3BD',
  'B0BRYFBCRF',
  'B0B1JCJNJ4',
  'B09Y4X2XTS',
  'B0BG2ZDW9P',
  'B09QWDNQH9'],
 'retrieved_context': ['The Complete Harry Potter Film Music Collection ',
  'If ',
  'Beatopia[LP] ',
  'Girls, Girls, Girls ',
  'Marry Me Soundtrack ',
  'Bluey Dance Mode Orange ',
  'IM NAYEON[NA ver.] ',
  'Songs About You ',
  'Love ',
  'Radiate Like This[LP] ']}

In [16]:
print(output["answer"])

I don’t have any earphones listed among the available products right now.


In [17]:
output = rag_pipeline("Do you have any earphones?", top_k=15)

In [18]:
print(output["answer"])

I don’t see any earphones listed among the available products.


In [19]:
output

{'data_object': RAGGenerationResponse(answer='I don’t see any earphones listed among the available products.', references=[]),
 'answer': 'I don’t see any earphones listed among the available products.',
 'references': [],
 'question': 'Do you have any earphones?',
 'retrieved_context_ids': ['B07VDPWPLQ',
  'B0C61QHRB6',
  'B09TMYXDMG',
  'B09ZTFPVNB',
  'B09PNST3BD',
  'B0BRYFBCRF',
  'B0B1JCJNJ4',
  'B09Y4X2XTS',
  'B0BG2ZDW9P',
  'B09QWDNQH9',
  'B09XBGM4GB',
  'B0B8DK9VH2',
  'B0BS1SHT91',
  'B0B1TPKCPB',
  'B0B72B3NRC'],
 'retrieved_context': ['The Complete Harry Potter Film Music Collection ',
  'If ',
  'Beatopia[LP] ',
  'Girls, Girls, Girls ',
  'Marry Me Soundtrack ',
  'Bluey Dance Mode Orange ',
  'IM NAYEON[NA ver.] ',
  'Songs About You ',
  'Love ',
  'Radiate Like This[LP] ',
  'Licked Live In NYC[2 CD] ',
  'The Loneliest Time[LP]       Explicit Lyrics ',
  "Now That's What I Call A Love Song / Various ",
  'Against The Odds: 1974-1982[8 CD] ',
  'The Mars Volta ']}